In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [8]:
# Define a small BMW Q&A dataset. Since this is a proof of concept, 5-10 examples will suffice.
qa_dataset = [
    {"q": "What serves as the headquarters of BMW?", "keywords": ["Munich", "Germany"]},
    {"q": "Which series is BMW's compact executive car?", "keywords": ["3 Series", "3-Series"]},
    {"q": "Does BMW produce electric vehicles?", "keywords": ["Yes", "i3", "iX", "electric"]},
    {"q": "What does BMW stand for?", "keywords": ["Bayerische", "Bavarian"]},
    {"q": "Is the BMW M3 a sports car?", "keywords": ["Yes", "performance", "sport"]}
]

MODEL_PATH_REDUCED = "models/bmw-gpt2-reduced/reduced_model"
MODEL_PATH_ORIGINAL = "models/bmw-gpt2-reduced/original_model"

In [9]:
def evaluate_model(model_path, dataset):
    print(f"\n--- Evaluating Model: {model_path} ---")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForCausalLM.from_pretrained(model_path)
    except Exception as e:
        print(f"Could not load model from {model_path}. Did you train it? Error: {e}")
        return

    model.eval()
    correct_count = 0
    
    for item in dataset:
        question = item['q']
        keywords = item['keywords']
        
        # Generate input: Simulate a simple question-and-answer format
        input_text = f"Question: {question}\nAnswer:"
        inputs = tokenizer(input_text, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=100, 
                pad_token_id=tokenizer.eos_token_id,
                temperature=0.7,
                top_k=50,
                top_p=0.95,
                do_sample=True
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract the Answer section
        answer_part = generated_text.split("Answer:")[-1].strip()

        # Simple evaluation criteria: Does the generated text contain the expected keywords?
        is_correct = any(k.lower() in answer_part.lower() for k in keywords)
        if is_correct:
            correct_count += 1
            
        print(f"Q: {question}")
        print(f"Gen: {answer_part}")
        print(f"Result: {'✅' if is_correct else '❌'}")
        print("-" * 20)

    accuracy = correct_count / len(dataset)
    print(f"Final Accuracy for {model_path}: {accuracy:.2%}")
    return accuracy

if __name__ == "__main__":
    # Evaluate original
    evaluate_model(MODEL_PATH_ORIGINAL, qa_dataset)
    # Evaluate reduced
    evaluate_model(MODEL_PATH_REDUCED, qa_dataset)


--- Evaluating Model: models/bmw-gpt2-reduced/original_model ---
Q: What serves as the headquarters of BMW?
Gen: It is the same building as the BMW Group. The building is located at the entrance to the parking lot and is located near the Munich International Airport. The BMW Group is based in the US.
The BMW Group is the company's largest car brand and is an innovative company. It provides both high-quality product and service in the automotive segment.
BMW Group is based in the Netherlands. It provides a full range of products and services in the automotive sector.
BMW Group is also
Result: ✅
--------------------
Q: Which series is BMW's compact executive car?
Gen: BMW has been in the same car for 10 years, and in the same car for over a decade. However, the BMW iX4 was the first to achieve the world's first hybrid and has been very successful.
BMW has been using the iX4 in its range of products since 1989. The car is a hybrid. It has been in production for over a decade and has been